# Q&A LLM Chatbot with RAG using Databricks

**Pipeline**

- Q&A chatbot with general knowledge (via `Mixtral-8x7B` Large Language Model) and specialized knowledge (using Retrieval Augmented Generation (RAG))
- Data engineering for LLM input data
- Databricks AI features include: Vector Search Endpoints, Serverless Foundation Model Endpoints, and Managed MLFlow
- Databricks data features include: Delta Tables, Unity Catalog, and Hive
- Ability to modify prompt template

**Chatbot Implementation**

![](chatbot_screenshot.png)

# 0. Set-Up Environment

## Authentication with Personal Access Token (PAT)
Generate a Personal Authentication Token (PAT): [PAT Instructions](https://docs.databricks.com/en/dev-tools/auth/pat.html)

There are two methods for supplying the newly created PAT to this notebook: 
1. Create a **Secret** (recommended)
2. Add it directly to notebook as a variable

## Initialize Environment
Install and import required packages

In [ ]:
%pip install \
    mlflow==2.10.2 \
    transformers==4.30.2 \
    langchain==0.1.8 \
    databricks-vectorsearch==0.22 \
    databricks-sdk==0.20.0 \
    markdownify

dbutils.library.restartPython()

In [ ]:
import os
import requests
from requests.adapters import HTTPAdapter
import xml.etree.ElementTree as ET
from concurrent.futures import ThreadPoolExecutor
from urllib3.util.retry import Retry
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.functions import col, udf, length, lit, pandas_udf
from pyspark.sql.types import StringType
import mlflow
import langchain

## Configuration

In [ ]:
##-- PAT Configuration
'''
If you're supplying the PAT as a Notebook Variable, edit the line 
below to add your PAT token. E.g.: token_param = "dapiabc123..."
If you've added the PAT as a secret, do not edit this cell.
'''
token_param = "PAT TOKEN"

### Unity Catalog Configuration
1. Create a new Databricks `catalog` 
2. Add the new `catalog` name below 

In [ ]:
##-- Unity Catalog Configuration
'''
Add the name of your catalog below
E.g.: catalog = "my_catalog"
'''
catalog = "hive_metastore"

### General Configuration
No changes needed here - these environmental parameters are intended to be static

In [ ]:
##-- Get Databricks username to prepend demo artifacts
email_param = spark.sql('select current_user() as user').collect()[0]['user']
username_param = email_param.split('@')[0].replace('.', '_')

##-- Unity Catalog Locations
db = f"{username_param}_docs_rag_demo"

##-- Pre/Post Processed Data Tables
cloud_platform = "aws"
bronze_data_table, silver_data_table = f"{cloud_platform}_docs_bronze_data", f"{cloud_platform}_docs_silver_data"
vs_index_table = silver_data_table + f"_{username_param}_index"

##-- Table Paths
bronze_data_table_path, silver_data_table_path, vs_index_table_path = f"{catalog}.{db}.{bronze_data_table}", f"{catalog}.{db}.{silver_data_table}", f"{catalog}.{db}.{vs_index_table}"

##-- Vector Search Table Configuration
endpoint_model_name = f"{username_param}_docs_rag_vs_endpoint"

##-- Embedding Configurations
embedding_model_endpoint = "databricks-bge-large-en"
embedding_text_data = "content"

##-- Endpoint Configurations
foundation_endpoint = "databricks-mixtral-8x7b-instruct"
hf_tokenizer_repo = "mistralai/Mixtral-8x7B-v0.1"

##-- Model Configurations
run_name = "chatbot_docs_rag"
model_name = f"{catalog}.{db}.{run_name}"

##-- Databricks Host for authentication
domain_param = spark.conf.get("spark.databricks.workspaceUrl")
host_param = "https://" + domain_param

### Credentials Test
Refer to the PAT instructions for credentials troubleshooting if needed

In [ ]:
from databricks.sdk.errors.platform import PermissionDenied

##-- Retrieve secret token if exists
secret_scope, secret_key = "rag_demo", "pat"

if secret_scope in (s.name for s in dbutils.secrets.listScopes()):
  if secret_key in (s.key for s in dbutils.secrets.list(secret_scope)):
    token_param = dbutils.secrets.get(secret_scope, secret_key)

##-- Test Token is supplied correctly
from databricks.sdk import WorkspaceClient
w = WorkspaceClient(
  host=host_param,
  token=token_param
)
try:
  wid = w.get_workspace_id()
except PermissionDenied as e:
  print("ERROR: The credentials you provided aren't working.")
  print("Please review the instructions above in the \"Configuration\" section and retry.")
  raise e

print(f"PAT authentication succeeded for workspace {wid}!")

# I. ETL Data - AWS

## Create Schema
Create a `schema` for the tables

In [ ]:
##-- Create Schemas and Tables
spark.sql(f'''
USE CATALOG {catalog}
''')

spark.sql(f'''
CREATE SCHEMA IF NOT EXISTS {db} 
''')

## Download Data and Write to Bronze Delta Table
Extract and load the data (URL and text from Amazon Bedrock documentation sitemap and pages) as HTML into the Unity Catalog table

In [ ]:
sitemap = "https://docs.aws.amazon.com/bedrock/latest/userguide/sitemap.xml"

In [ ]:
##-- This function downloads the AWS documentation from the web
def download_aws_documentation_articles(
  sitemap,
  max_documents = None,
  retries=Retry(total=3, backoff_factor=3,status_forcelist=[429])
  ):
  def grab_sitemap_urls(sitemap: str):
  # Fetch the XML content from sitemap
    response = requests.get(sitemap)
    root = ET.fromstring(response.content)

    # Find all 'loc' elements (URLs) in the XML
    urls = [loc.text for loc in root.findall(".//{http://www.sitemaps.org/schemas/sitemap/0.9}loc")]
    if max_documents:
      urls = urls[:max_documents]
    return urls

  ##-- Pandas UDF to fetch HTML content for a batch of URLs
  @pandas_udf("string")
  def fetch_html_udf(urls: pd.Series) -> pd.Series:
    ##-- retries with backoff
    adapter = HTTPAdapter(max_retries=retries)
    http = requests.Session()
    http.mount("http://", adapter)
    http.mount("https://", adapter)
    def fetch_html(url):
      try:
        response = http.get(url)
        if response.status_code == 200:
          return response.content
      except requests.RequestException:
        return None
      return None

    with ThreadPoolExecutor(max_workers=200) as executor:
      results = list(executor.map(fetch_html, urls))
    return pd.Series(results)
  
  ##-- Get the URLs for each page on sitemap
  urls = grab_sitemap_urls(sitemap)
  ##-- Create DataFrame from URLs
  df_urls = spark.createDataFrame(urls, StringType()).toDF("url").repartition(10)
  ##-- Add sitemap column
  df_with_sitemap = df_urls.withColumn("sitemap", lit(sitemap))
  ##-- Apply UDFs to DataFrame
  df_with_html = df_with_sitemap.withColumn("text", fetch_html_udf("url"))

  return df_with_html

In [ ]:
def extract_data(bronze_data_table_path):

    if (spark.catalog.tableExists(bronze_data_table_path) and 
        not spark.table(bronze_data_table_path).isEmpty()):
        print(f"Table is already populated: {bronze_data_table_path}")
        return display(spark.table(bronze_data_table_path).limit(2))
    else:
        print(f"Table needs to be populated: {bronze_data_table_path}")
        try:
            print("Downloading data . . . \n")
            df_docs = download_aws_documentation_articles(sitemap)
            print("Saving table . . . \n")
            (df_docs.write
                .format("delta")
                .mode('overwrite')
                .option("overwriteSchema", "true")
                .saveAsTable(bronze_data_table_path))
            
            print(f"Table created: {bronze_data_table_path}")
            return display(spark.table(bronze_data_table_path).limit(2))
        except Exception as e:
            raise Exception(f"Issue downloading table. See error:\n {e}")
        
extract_data(bronze_data_table_path)


## Parse and Split Data

This code parses out text and example code from the html pages in the bronze table and splits it into smaller chunks for retrieval by the vector search engine. 

In [ ]:
from markdownify import markdownify as md
from bs4 import BeautifulSoup
from langchain.text_splitter import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from transformers import AutoTokenizer

max_chunk_size, min_chunk_size = 700, 2

In [ ]:
def split_awsdoc_into_chunks(html: str):
  soup = BeautifulSoup(html, "html.parser")
  article_soup = soup.find(id="main-content")
  article_soup.find("noscript").decompose() ##-- omit no javascript warning
  article_soup.find(id="main-col-footer").decompose() ##-- remove footer

  article_html = str(article_soup).strip()
  ##-- Convert to MD because to preserve code - Langchain's HTML text splitter strips code blocks
  article_md = md(article_html.strip(), heading_style="ATX")

  headers_to_split_on = [
      ("#", "Header 1"),
      ("##", "Header 2"),
      ("###", "Header 3"),
  ]

  def adjust_chunk_sizes(in_chunks):
    if not html:
        return []
      
    tokenizer = AutoTokenizer.from_pretrained(hf_tokenizer_repo)
    
    text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
      tokenizer, 
      chunk_size=max_chunk_size, 
      chunk_overlap=0,
      strip_whitespace=True
    )

    chunks = []
    hold_chunk = ""
    
    ##-- Adjust chunk sizes
    for c in in_chunks:
      if len(tokenizer.encode(c.page_content)) < min_chunk_size:
        continue # skip small chunks
      
      ##-- Serialize chunk content
      def md_headers(metadata):
        h1_line = f"# {metadata['Header 1']} \n" if "Header 1" in metadata else ""
        h2_line = f"## {metadata['Header 2']} \n" if "Header 2" in metadata else ""
        h3_line = f"### {metadata['Header 3']} \n" if "Header 3" in metadata else ""
        return h1_line + h2_line + h3_line
      
      content = hold_chunk + md_headers(c.metadata) + c.page_content.strip()
      
      ##-- Split chunk if too large
      if len(tokenizer.encode(content)) > max_chunk_size:
        
        ##-- Add hold chunk
        if hold_chunk != "":
          chunks.append(hold_chunk.strip())
          hold_chunk = ""
        
        new_chunks = text_splitter.split_text(c.page_content.strip())
        chunks.extend(md_headers(c.metadata) + chunk for chunk in new_chunks)

      ##-- Append chunk to next if too small
      elif len(tokenizer.encode(content)) < max_chunk_size // 2:
        hold_chunk += content + "\n\n"
      
      ##-- Add chunk to main list
      else:
          chunks.extend(text_splitter.split_text(content.strip()))
          hold_chunk = ""
    
    ##-- Add final hold chunk
    if hold_chunk != "":
        chunks.append(hold_chunk.strip())
    return chunks

  html_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
  raw_chunks = html_splitter.split_text(article_md)
  chunks = adjust_chunk_sizes(raw_chunks)
  return chunks

In [ ]:
html_df = spark.table(bronze_data_table_path)
html_list = [row[0] for row in html_df[["text"]].collect()]

chunks = []

##-- Split each page and load into text chunks
for page in html_list:
  chunks += split_awsdoc_into_chunks(page)

## Process Data and Write to Silver Delta Table
This table includes the processed AWS documentation data. It is separated into small text chunks.

In [ ]:
##-- Create user-defined function (UDF) to parse documents w/ Spark 
@pandas_udf("array<string>")
def parse_and_split(docs: pd.Series) -> pd.Series:
    return docs.apply(split_awsdoc_into_chunks)
    
(spark.table(bronze_data_table_path)
      .filter('text is not null')
      .withColumn(embedding_text_data, F.explode(parse_and_split('text')))
      .drop("text", "sitemap")
      .write
      .option("overwriteSchema", "true")
      .option("delta.enableChangeDataFeed", "true")
      .mode('overwrite')
      .saveAsTable(silver_data_table_path)
      )

display(spark.table(silver_data_table_path))

# II. Create Vector Search

## Create Vector Search Endpoint
Creating a new Databricks Vector Search Endpoint can take a long time. If the request below times out, you can check the [Vector Search Endpoints UI](#/setting/clusters/vector-search) to check on the status of your endpoint and resume once the endpoint is ready.

In [ ]:
from databricks.sdk.service.vectorsearch import EndpointType, PipelineType
from datetime import timedelta

endpoint_names = [ep.name for ep in w.vector_search_endpoints.list_endpoints()]
if endpoint_model_name not in endpoint_names:
    ##-- Create Vector Search Endpoint
    print("Creating a vector search endpoint. This may take up to 60 minutes...")

    w.vector_search_endpoints.create_endpoint_and_wait(
        name=endpoint_model_name,
        endpoint_type=EndpointType.STANDARD,
        timeout=timedelta(minutes=60)
    )
else:
    ##-- If endpoint already exists, skip
    print(f"Vector Search Endpoint \"{endpoint_model_name}\" already exists. skipping...")

## Create Vector Search Index

In [ ]:
from databricks.sdk.service.vectorsearch import DeltaSyncVectorIndexSpecRequest, PipelineType, VectorIndexType, EmbeddingConfig
from dataclasses import dataclass
import time

@dataclass
class EmbeddingSourceColumn:
  """EmbeddingSourceColumn class to fix a bug in databricks-sdk v0.20.0"""
  embedding_model_endpoint_name: str = None
  name: str = None

  def as_dict(self) -> dict:
    body = {
      "embedding_model_endpoint_name":self.embedding_model_endpoint_name,
      "name":self.name
    }
    return body

def wait_until_index_ready(index, max_attempts=20, wait_seconds=20):
  for _ in range(max_attempts):
    status = w.vector_search_indexes.get_index(index_name=index).status
    success_message = "Index creation succeeded"
    if status.ready and status.message.startswith(success_message): break
    print("Waiting for index to become ready...")
    time.sleep(wait_seconds)


indexes = (i.name for i in w.vector_search_indexes.list_indexes(endpoint_model_name))
if vs_index_table_path not in indexes:
  w.vector_search_indexes.create_index(
    name=vs_index_table_path,
    endpoint_name=endpoint_model_name,
    index_type=VectorIndexType.DELTA_SYNC,
    primary_key="url",
    delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
      source_table=silver_data_table_path,
      pipeline_type=PipelineType.TRIGGERED,
      embedding_source_columns=[EmbeddingSourceColumn(
        name=embedding_text_data, 
        embedding_model_endpoint_name=embedding_model_endpoint
      )]
    )
  )
  wait_until_index_ready(vs_index_table_path)
  print(f"Index {vs_index_table_path} created.")

else:
  print("Index already exists. Syncing index instead.")
  wait_until_index_ready(vs_index_table_path) ##-- wait in case previous sync isn't yet complete
  w.vector_search_indexes.sync_index(index_name=vs_index_table_path)
  wait_until_index_ready(vs_index_table_path)
  print("Sync complete.")

# III. Deploy Q&A Chatbot Model w/ RAG

## Build Langchain Retriever

In [ ]:
from databricks.vector_search.client import VectorSearchClient
from langchain.vectorstores import DatabricksVectorSearch
from langchain.embeddings import DatabricksEmbeddings

##-- Setup authentication for model

##-- Test embedding Langchain model
##-- NB: Question embedding model must match the one used in the chunk in the previous model 
embedding_model = DatabricksEmbeddings(endpoint=embedding_model_endpoint)
print(f"Test embeddings: {embedding_model.embed_query('What is Amazon Bedrock?')[:20]}...")

def get_retriever(persist_dir: str = None,
                  db_vs_index_table_path="",
                  db_endpoint_model_name="",
                  db_embedding_model_endpoint="",
                  db_embedding_text_data="",
                  db_databricks_token="",
                  db_host=""):
    
    ##-- Get the serverless endpoint URL used to send requests to the model 
    vsc = VectorSearchClient(
        workspace_url=db_host, 
        personal_access_token=db_databricks_token
        )
    
    ##-- Retrieve the index
    vs_index = vsc.get_index(
        endpoint_name=db_endpoint_model_name,
        index_name=db_vs_index_table_path
        )

    ##-- Create the retriever
    vectorstore = DatabricksVectorSearch(
        vs_index, 
        text_column=db_embedding_text_data, 
        embedding=db_embedding_model_endpoint
        )
    return vectorstore.as_retriever()

In [ ]:
##-- Test retriever
vectorstore = get_retriever(
    db_vs_index_table_path=vs_index_table_path,
    db_endpoint_model_name=endpoint_model_name,
    db_embedding_model_endpoint=embedding_model_endpoint,
    db_embedding_text_data=embedding_text_data,
    db_databricks_token=token_param,
    db_host=host_param
    )

similar_documents = vectorstore.get_relevant_documents("How do I list foundation models?")
print(f"Relevant documents: {similar_documents[0]}")

## Build Chat Model
Use a LangChain Chat Model to query the Foundation Model

In [ ]:
##-- Test Databricks Foundation LLM model
from langchain.chat_models import ChatDatabricks
chat_model = ChatDatabricks(endpoint=f"{foundation_endpoint}", max_tokens = 200)
print(f"Test chat model:\n {chat_model.invoke('What is Amazon Bedrock')}")

## Assemble RAG Chain

In [ ]:
template = '''

You are an AWS engineer/architect who helps customers 
answer questions about AWS and Amazon services. 
You answer questions involving (but not limited to): 

Service features, Basic How-tos, API, Cloud, Python, SQL, 
coding, Machine Learning, data engineering, data science, 
data warehouses, security, services, and infrastructure. 

If the question is not related to one of these topics, kindly decline to answer. 

If you don't know the answer, just say that you don't know, don't try to make up an answer. 

Keep the answer as concise as possible. 

Use the following pieces of context to answer the question at the end: {context} 

Question: {question}
Answer: 

'''

In [ ]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.chat_models import ChatDatabricks

def create_chain(docs_template):
  prompt = PromptTemplate(template=docs_template, input_variables=["context", "question"])
  chain = RetrievalQA.from_chain_type(
    llm=chat_model,
    chain_type="stuff",
    retriever=vectorstore,
    chain_type_kwargs={"prompt": prompt}
    )
  return chain

chain = create_chain(docs_template=template)

## Assemble Chatbot

In [ ]:
from IPython.display import DisplayObject, TextDisplayObject

##-- Use Markdown to pretty-print LLM responses
class Markdown(TextDisplayObject):
  def __init__(self,TextDisplayObject):
    import markdown as md
    self.html = md.markdown(TextDisplayObject)

  def _repr_html_(self):
    return self.html

QA_chatbot = lambda question: chain.invoke({"query": question})

def run_QA_chatbot(questions,chatbot=QA_chatbot):
    display(Markdown(f"# Amazon Bedrock Q&A"))
    for q in questions:
        answer = chatbot(q)
        markdown_output = Markdown(f'''### Query:\n\n{answer["query"]}\n\n### Answer from LLM:\n\n{answer["result"]}\n\n''')
        display(markdown_output)
    return

## Test Chatbot

This section allows the user to input custom questions and returns with readable answers

In [ ]:
##-- Questions to ask the LLM - feel free to replace these with your own!
questions = ["What is Amazon Bedrock?"
             "Which models can I use with Bedrock Agents?",
             "What are Bedrock agents?",
             "How can I keep my data secure while using it with Bedrock?",
             "What is the cost of using Bedrock?",
             "What AWS products intergrate with Bedrock?",
             "What makes Bedrock unique compared to other generative AIs?",
             "What skillsets do I need on my team to effectively fine-tune a Bedrock model and evaluate its performance?",
             "What are some features of Amazon Bedrock?",
             "How do I get information about Bedrock foundation models?",
             "What are Amazon Bedrock playgrounds?",
             "What is a foundation model in Amazon Bedrock?",
             "What types of models are Bedrock playgrounds available for?"
]

In [ ]:
run_QA_chatbot(questions)

## Save Model to Registry
Save Model to MLflow Model Registry in Unity Catalog. This allows you to track model versions and reference the model in other notebooks or within serving endpoints

In [ ]:
from mlflow.models import infer_signature

mlflow.set_registry_uri("databricks-uc")

question = {"query": "What is Amazon Bedrock?"}
answer = chain.invoke(question)

with mlflow.start_run(run_name=run_name) as run:
    signature = infer_signature(question, answer)
    model_info = mlflow.langchain.log_model(
        chain,
        loader_fn=get_retriever,
        artifact_path="chain",
        registered_model_name=model_name,
        pip_requirements=[
            "mlflow==" + mlflow.__version__,
            "langchain==" + langchain.__version__,
            "databricks-vectorsearch==0.22",
        ],
        input_example=question,
        signature=signature
    )